# Lecture 2 — Class Exercise
# Bar Charts: World Happiness Report 2023

---

> **Your task:** Create **2 polished bar charts** using the World Happiness Report dataset.  
> **Push to:** `week02/lecture02_exercise.ipynb` in **your own GitHub repo** before the end of class.

---

### Rules to keep in mind 💡
1. Every bar chart **must have a zero baseline** — no exceptions
2. Every chart **must have an insight title**, not a topic title
3. Aim for **professional quality** — clean background, readable font, no clutter
4. Horizontal bars for long category names

---


## Setup — Run this cell first


In [8]:
import pandas as pd
import numpy as np
import os

# World Happiness Report 2023
# Source: https://www.kaggle.com/datasets/ajaypalsinghlo/world-happiness-report-2023

# Tries your local path first, then falls back to the CSV in the same folder
local_path = ('/Users/vaibhavmalhotra/Desktop/Data_Visualization/data-viz-class-material-w2026/data/world_happiness_2023.csv')
path = local_path if os.path.exists(local_path) else 'world_happiness_2023.csv'

df = pd.read_csv(path)
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']

print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())

Dataset: 63 countries, 9 columns
       Country                        Region  Happiness_Score     GDP  \
0      Finland                Western Europe            7.804  10.775   
1      Denmark                Western Europe            7.586  10.933   
2      Iceland                Western Europe            7.525  10.878   
3       Israel  Middle East and North Africa            7.473  10.527   
4  Netherlands                Western Europe            7.464  11.015   

   Social_Support  Life_Expectancy  Freedom  Generosity  Corruption  
0           0.954             71.9    0.949       0.142       0.179  
1           0.954             72.7    0.931       0.168       0.234  
2           0.983             72.5    0.961       0.260       0.150  
3           0.916             72.4    0.903       0.149       0.826  
4           0.939             72.4    0.879       0.240       0.296  


In [9]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])


Regions in dataset:
Region
Western Europe                  15
Latin America and Caribbean     13
Central and Eastern Europe       7
Sub-Saharan Africa               7
Middle East and North Africa     6
North America and ANZ            4
Southeast Asia                   4
South Asia                       4
East Asia                        3
Name: count, dtype: int64

Score range: 1.859 – 7.804

Bottom 10 countries:
        Country                        Region  Happiness_Score
60  Afghanistan                    South Asia            1.859
61      Lebanon  Middle East and North Africa            2.392
62     Zimbabwe            Sub-Saharan Africa            2.995
52     Ethiopia            Sub-Saharan Africa            3.564
53     Tanzania            Sub-Saharan Africa            3.698
48   Bangladesh                    South Asia            3.892
47        India                    South Asia            4.036
50        Kenya            Sub-Saharan Africa            4.112
54       Uganda

---
## Task 1 — Regional Comparison Bar Chart

**What to build:** A horizontal bar chart showing the **average happiness score by region**, sorted from highest to lowest.

**Requirements:**
- Horizontal orientation (category names are long)
- Sorted by score, descending (so the happiest region is at the top)
- Zero baseline on x-axis
- At least one design choice that goes beyond the Plotly default (colour, annotation, labels, etc.)
- An insight title that answers: *which region stands out and why does it matter?*

**Hint:** Use `df.groupby('Region')['Happiness_Score'].mean()` to compute the averages.


In [10]:
# Task 1: Regional comparison bar chart
# -------------------------------------

# Step 1: Compute average happiness score by region
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score'))  # ascending -> happiest ends up at the top of a horizontal bar

print(region_avg)

# Step 2: Colour logic -> highlight the leaders (blue) and the lowest region (orange), rest grey
top_region = region_avg.iloc[-1]['Region']
low_region = region_avg.iloc[0]['Region']
top_one = region_avg.tail(1)['Region'].tolist()

def pick_colour(r):
    if r in top_one:  return '#2E75B6'
    if r == low_region: return '#E07B39'
    return '#BFC5CC'

colours = [pick_colour(r) for r in region_avg['Region']]

# Step 3: Build the chart
fig = go.Figure(go.Bar(
    x=region_avg['Happiness_Score'],
    y=region_avg['Region'],
    orientation='h',
    marker=dict(color=colours, line_width=0),
    text=region_avg['Happiness_Score'].round(2),
    textposition='outside',
    cliponaxis=False,
    hovertemplate='%{y}: %{x:.2f}<extra></extra>'
))

gap = region_avg['Happiness_Score'].max() / region_avg['Happiness_Score'].min()

fig.update_layout(
    title=dict(text=f'Western Europe leads at 7.09, while South Asia scores less than half ~ {region_avg["Happiness_Score"].min():.1f}',
               x=0.0, font=dict(size=18)),
    xaxis=dict(range=[0, 8.5], title='Average Happiness Score (0–10)', gridcolor='#EEEEEE', zeroline=True, zerolinecolor='#999999'),
    yaxis=dict(title='', gridcolor='white'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    showlegend=False,
    margin=dict(l=10, r=40, t=70, b=50),
    height=520
)
fig.add_annotation(
    x=region_avg['Happiness_Score'].min(), y=low_region,
    text=f'  Top region is {gap:.1f}x higher',
    showarrow=False, xanchor='left', xshift=45,
    font=dict(color='#E07B39', size=12)
)
fig.show()

                         Region  Happiness_Score
5                    South Asia         3.618250
7            Sub-Saharan Africa         4.064714
3  Middle East and North Africa         4.943333
6                Southeast Asia         5.695250
2   Latin America and Caribbean         5.699000
1                     East Asia         5.966000
0    Central and Eastern Europe         6.338143
4         North America and ANZ         7.018250
8                Western Europe         7.085533


---
## Task 2 — Bottom vs. Top: A Contrast Story

**What to build:** A bar chart that highlights the **gap between the happiest and least happy countries**, focusing on a specific insight.

**Requirements:**
- Show the **top 8 AND bottom 8 countries** together (16 bars total)
- Use **colour** to distinguish the two groups (not Plotly's default rainbow)
- Add a **visual separator or annotation** that emphasises the gap
- Insight title that tells the story of the gap

**Hint:** Use `pd.concat([df.nlargest(8,'Happiness_Score'), df.nsmallest(8,'Happiness_Score')])` to get both groups.

**Stretch goal:** Add a vertical reference line showing the global average.


In [11]:
# Task 2: Top 8 vs. Bottom 8 contrast
# ------------------------------------

# Step 1: Get top and bottom countries
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')   # ascending -> happiest at top
global_avg = df['Happiness_Score'].mean()
top_avg, bottom_avg = top8['Happiness_Score'].mean(), bottom8['Happiness_Score'].mean()
print(f"Global average: {global_avg:.2f} | Top 8 avg: {top_avg:.2f} | Bottom 8 avg: {bottom_avg:.2f}")

# Step 2: Build your chart
colour_map = {'Top 8': '#2E75B6', 'Bottom 8': '#E07B39'}

fig = go.Figure(go.Bar(
    x=combined['Happiness_Score'],
    y=combined['Country'],
    orientation='h',
    marker=dict(color=combined['Group'].map(colour_map), line_width=0),
    text=combined['Happiness_Score'].round(2),
    textposition='outside',
    cliponaxis=False,
    hovertemplate='%{y}: %{x:.2f}<extra></extra>'
))

fig.update_traces(marker_line_width=0)

fig.update_layout(
    title=dict(text=f'The happiest countries score more than double the least happy ({top_avg:.1f} vs {bottom_avg:.1f} on average)',
               x=0.0, font=dict(size=18)),
    xaxis=dict(range=[0, 8.5], title='Happiness Score (0–10)', gridcolor='#EEEEEE', zeroline=True, zerolinecolor='#999999'),
    yaxis=dict(gridcolor='white', title=''),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    margin=dict(l=10, r=40, t=70, b=50),
    showlegend=False,
    height=650
)

# Visual separator between the two groups (categorical axis: bars sit at 0..15, gap is at 7.5)
fig.add_hline(y=7.5, line_dash='dot', line_color='#888888', line_width=1.5)

# Stretch goal: global average reference line
fig.add_vline(x=global_avg, line_dash='dash', line_color='#444444', line_width=1.5,
              annotation_text=f'Global avg {global_avg:.2f}', annotation_position='top',
              annotation_font=dict(size=11, color='#444444'))

# Annotations: colour legend + the gap
fig.add_annotation(x=8.4, y=11.5, text='Top 8 (blue)', showarrow=False, xanchor='right',
                   font=dict(color='#2E75B6', size=12))
fig.add_annotation(x=8.4, y=3.5, text='Bottom 8 (orange)', showarrow=False, xanchor='right',
                   font=dict(color='#E07B39', size=12))
fig.add_annotation(x=8.4, y=7.5, text=f'Gap: {top_avg - bottom_avg:.1f} points', showarrow=False, xanchor='right',
                   yshift=0, font=dict(size=12, color='#444444'),
                   bgcolor='white', bordercolor='#888888', borderwidth=1, borderpad=4)

fig.show()

Global average: 5.81 | Top 8 avg: 7.48 | Bottom 8 avg: 3.32


---
## Done? Stretch Goal

If you finish both tasks with time to spare, try this:

**Task 3 (stretch):** Build a **grouped bar chart** comparing 2 sub-factors (e.g. `GDP_per_capita` and `Freedom`) across the 5 most populated regions. Use colour meaningfully and write an insight title.

Regions to include: `'Western Europe'`, `'Latin America'`, `'East Asia'`, `'Sub-Saharan Africa'`, `'South Asia'`


In [12]:
# Stretch goal — grouped bar chart
# Note: Logged GDP (~7-11) and Freedom (0-1) are on different scales, so putting them on one axis
# would be misleading. Social Support and Freedom are both 0-1 shares, so they compare fairly.

regions = ['Western Europe', 'Latin America and Caribbean', 'East Asia',
           'Sub-Saharan Africa', 'South Asia']   # 'Latin America' is named 'Latin America and Caribbean' in this dataset

grp = (df[df['Region'].isin(regions)]
       .groupby('Region')[['Social_Support', 'Freedom']].mean()
       .reindex(regions)
       .reset_index())
print(grp.round(3))

fig = go.Figure()
fig.add_bar(name='Social Support', x=grp['Region'], y=grp['Social_Support'],
            marker_color='#2E75B6', text=grp['Social_Support'].round(2), textposition='outside')
fig.add_bar(name='Freedom to make life choices', x=grp['Region'], y=grp['Freedom'],
            marker_color='#F2A541', text=grp['Freedom'].round(2), textposition='outside')

fig.update_layout(
    barmode='group',
    title=dict(text='South Asia and Sub-Saharan Africa trail Western Europe by 0.3–0.4 points on both social support and freedom',
               x=0.0, font=dict(size=17)),
    yaxis=dict(range=[0, 1.1], title='Average score (0–1)', gridcolor='#EEEEEE', zeroline=True, zerolinecolor='#999999'),
    xaxis=dict(title=''),
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=13),
    legend=dict(orientation='h', y=1.08, x=0),
    margin=dict(l=10, r=30, t=90, b=50),
    height=520
)
fig.update_traces(marker_line_width=0, cliponaxis=False)
fig.show()

                        Region  Social_Support  Freedom
0               Western Europe           0.932    0.896
1  Latin America and Caribbean           0.829    0.754
2                    East Asia           0.853    0.754
3           Sub-Saharan Africa           0.558    0.599
4                   South Asia           0.561    0.546
